# 04 — XGBoost: training, tuning and feature importance

**Owner:** Shubhaang Kataruka · **Phase:** Advanced Machine Learning · **Project 21**

**Objective.** Predict whether a flight will arrive at least **15 minutes late**, using *only* permitted pre-departure attributes and strictly historical observations. This notebook runs the code in `src/flight_delay/models/xgboost_model.py` without copying modeling logic.

**Your deliverables:** untuned baseline; controlled feature-set comparison; train-only chronological hyperparameter search; validation AP/ROC-AUC; trained model; feature-importance table; row-aligned validation probabilities for Lakshya.

**Evaluation rule:** The **test set is not inspected in this notebook**. It is reserved for a one-time final evaluation after model selection and calibration methodology are frozen. Validation scores should not be described as unbiased test scores.

## 1. Environment and repository paths

From the repository root, set up a Python environment first:

```bash
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install -e .
python -m pip install ipykernel
python -m ipykernel install --user --name flight-delay-aml --display-name "Python (flight-delay AML)"
```

Select the corresponding kernel in Jupyter. No changes to your teammates' files are needed.

In [ ]:
import sys
from pathlib import Path


def repo_root(marker: str = "pyproject.toml") -> Path:
    start = Path.cwd().resolve()
    for current in (start, *start.parents):
        if (current / marker).exists():
            return current
    raise FileNotFoundError(f"Run from within the repository containing {marker}")

ROOT = repo_root()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
DATA_PATH = ROOT / "data" / "raw" / "flights.csv"
RESULTS_DIR = ROOT / "reports" / "results"
MODEL_DIR = ROOT / "models" / "artifacts"
assert DATA_PATH.exists(), f"Missing the 2013 flights dataset: {DATA_PATH}"
print("Repository:", ROOT)
print("Dataset:", DATA_PATH, f"({DATA_PATH.stat().st_size:,} bytes)")

## 2. Set an explicit experiment budget

The default is **8 randomized candidates × 3 chronological date-based CV folds**, plus full-train refitting and 3 feature-set baselines. Start with `N_ITER=2` for a smoke test; return to at least 8 candidates for the reported experiment. Use `N_JOBS=4` to avoid overloading a student laptop.

The CV date gap is one calendar date. **Important qualification:** the upstream rolling historical features use earlier *flight dates*, not independently verified outcome-availability timestamps. The team must document this limitation and validate it if production decision-time correctness is required.

In [ ]:
SEED = 42
N_ITER = 8        # Change to 2 for a quick end-to-end smoke test
CV_FOLDS = 3
N_JOBS = 4
RUN_TUNING = True

from flight_delay.models.xgboost_model import FEATURE_SETS, run_experiments

for name, features in FEATURE_SETS.items():
    print(f"{name:9} {len(features):2d} pre-departure columns")

## 3. Run the model experiments

The underlying runner loads the original data pipeline and:

1. Fits one untuned XGBoost baseline for each of `schedule`, `route`, and `full` feature sets.
2. Tunes the **full** feature set using **training dates only**, using Average Precision for CV ranking.
3. Compares probability-ranking metrics on the chronological **validation** partition.
4. Saves the best validation model and a stable, row-aligned prediction CSV.

No scaling is necessary for boosted decision trees. Numeric NaNs are filled with a sentinel; categorical features are one-hot encoded with unseen-category handling. The fitted transformations live **inside** each training pipeline, so they do not learn from validation or test rows.

In [ ]:
results = run_experiments(
    data_path=DATA_PATH,
    output_dir=RESULTS_DIR,
    model_dir=MODEL_DIR,
    n_iter=N_ITER,
    cv_folds=CV_FOLDS,
    random_state=SEED,
    n_jobs=N_JOBS,
    run_tuning=RUN_TUNING,
)
print("Selected experiment:", results["winner"])
display(results["leaderboard"].round(4))

## 4. Quantitative comparison (validation only)

- **Average Precision (AP)**: the primary ranking metric for delayed flights (~26% positives in the training set). AP differs mathematically from trapezoidal PR-AUC; label it precisely.
- **ROC-AUC**: complementary ranking metric.
- **Thresholded precision, recall, F1, Brier/calibration**: coordinated with Lakshya after raw probabilities are delivered.

Because the best experiment is selected using this validation set, final unbiased assessment requires the untouched test set.

In [ ]:
import matplotlib.pyplot as plt

leaderboard = results["leaderboard"].sort_values("average_precision")
ax = leaderboard.plot.barh(
    x="experiment", y="average_precision", legend=False, figsize=(9, 4),
)
ax.set(xlabel="Validation Average Precision (higher is better)", ylabel="Experiment")
plt.tight_layout()
plt.show()

print("Best validation experiment:", results["winner"])
print("Validation AP:", round(float(results["leaderboard"].iloc[0]["average_precision"]), 4))
print("Note: these metrics are NOT final test performance.")

## 5. Feature importance

The model's reported `gain_importance` measures reduction in the XGBoost tree objective at splits. With one-hot encoding, a source field such as `route` becomes multiple encoded features. Importance is **not causal effect**, and correlated predictors can share or exchange importance.

In [ ]:
import pandas as pd

importance = pd.read_csv(RESULTS_DIR / "xgboost_feature_importance.csv")
display(importance.head(20))
ax = importance.head(20).sort_values("gain_importance").plot.barh(
    x="encoded_feature", y="gain_importance", figsize=(10, 7), legend=False,
)
ax.set(xlabel="XGBoost gain importance", ylabel="Encoded model feature")
plt.tight_layout()
plt.show()

## 6. Validation predictions — team handoff

**Output:** `reports/results/xgboost_validation_predictions.csv` containing:

| Column | Meaning |
|---|---|
| `flight_id` | Original dataset ID for stable joins. Never trained as a feature. |
| `flight_date` | Chronological validation date. |
| `y_true` | Binary arrival-delay target (0 or 1). |
| `y_probability` | Predicted probability of a ≥15-minute arrival delay. |
| `model_name` | `xgboost` for model comparison. |
| `experiment` | Winning feature-set/model variant. |

**To Lakshya:** use these uncalibrated probabilities for validation plots and threshold selection. Coordinate the calibration fitting protocol to avoid fitting a calibrator on the same validation rows later used to judge it. Preserve the untouched final test partition.

In [ ]:
from flight_delay.evaluation.predictions import load_prediction_csv
predictions = load_prediction_csv(RESULTS_DIR / "xgboost_validation_predictions.csv")
assert predictions["flight_id"].is_unique
assert predictions["y_probability"].between(0, 1).all()
print("Validation prediction rows:", len(predictions))
display(predictions.head(8))

import json
manifest = json.loads((RESULTS_DIR / "xgboost_manifest.json").read_text())
print("Model:", MODEL_DIR / "xgboost_pipeline.joblib")
print("Chosen parameters:")
print(json.dumps(manifest["selected_model_parameters"], indent=2, default=str)[:2500])

## 7. Interpretation and report checklist

Document the following in your AML report / viva:

- Dataset and binary target; number of labeled rows and class proportions.
- No post-departure feature can pass through the **explicit model allowlist**.
- Historical rates use prior calendar dates (rolling daily evaluation; availability caveat).
- Train/validation/test are chronological; CV is date-grouped and trained on the training partition only.
- Both `scale_pos_weight=1` and class-ratio weighting were candidate hyperparameters.
- Three feature sets were compared; report their **validation AP and ROC-AUC** (do not invent results).
- The winning model, selected hyperparameters, top features and limitations.
- Pass raw probabilities to Lakshya for calibration/threshold selection.
- The final held-out test results can only be added after the team's entire model-selection process is frozen.

**Version control note:** the repository `.gitignore` deliberately ignores model artifacts and `reports/results/`. Share exported outputs through an agreed artifact mechanism; committing only the `.py` and `.ipynb` will not automatically commit your trained model or CSV results.